In [0]:
# Imports e preparação dos endereços das tabelas

from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window
import dateutil.parser
import requests
import pandas as pd

catalog = "atividade_2"

bronze = "bronze"
silver = "silver"
gold = "gold"

bronze_schema = f"{catalog}.{bronze}"
silver_schema = f"{catalog}.{silver}"
gold_schema = f"{catalog}.{gold}"

landing_path = f"/Volumes/{catalog}/{bronze}/Landing"

## Pergunta 1 - Receita total em R$

In [0]:
df_fact_movies_performance = spark.table(f"{gold_schema}.fact_movies_performance")

receita_total = df_fact_movies_performance.select(F.sum("receita_brl")).collect()[0][0]

print(f"Receita total dos filmes da base de dados: {receita_total:,}")

Receita total dos filmes da base de dados: 837,341,689,874.41


## Pergunta 2 - 5 filmes mais populares

In [0]:
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

filmes_mais_populares = (
    df_fact_movies_performance
        .join(df_dim_movies, on="sk_movie_id", how="inner")
        .orderBy(F.desc("popularidade"))
        .limit(5)
        .select("titulo", "popularidade")
)

display(filmes_mais_populares)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


## Pergunta 3 - Quantidade de filmes por gênero

In [0]:
df_dim_generos = spark.table(f"{gold_schema}.dim_generos")
df_bridge_movie_genre = spark.table(f"{gold_schema}.bridge_movie_genre")

count_generos = (
    df_bridge_movie_genre
        .join(df_dim_generos, "sk_genre_id")
        .groupBy("nome_genero")
        .agg(F.count("sk_movie_id").alias("volume"))
        .orderBy(F.desc("volume"))
)

display(count_generos)

nome_genero,volume
Drama,32542
Documentary,19055
Comedy,18903
Thriller,10399
Horror,9878
Romance,7721
Action,6250
Crime,4812
Animation,4588
TV Movie,4130


## Pergunta 4 - 10 filmes de maior receita

In [0]:
df_fact_movies_performance = spark.table(f"{gold_schema}.fact_movies_performance")

df_10_maiores_receitas = (
    df_fact_movies_performance
        .orderBy(F.desc("receita_brl"))
        .limit(10)
        .join(df_dim_movies, on="sk_movie_id", how="inner")
        .select(
            F.rank().over(Window.orderBy(F.desc("receita_brl"))).alias("posição"),
            "titulo",
            "receita_brl",
            "receita_usd"
        )
)

display(df_10_maiores_receitas)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posição,titulo,receita_brl,receita_usd
1,Avengers: Endgame,13981800000.00,2800000000.00
2,Avatar: The Way of Water,11586169778.17,2320250281.00
3,AVENGERS: INFINITY WAR,10248734497.25,2052415039.00
4,spider-man: no way home,9596743548.78,1921847111.00
5,The Lion King,8304567014.89,1663075401.00
6,Top Gun: Maverick,7433987341.66,1488732821.00
7,Barbie,7133439597.32,1428545028.00
8,The Super Mario Bros. Movie,6769814100.79,1355725263.00
9,Black Panther,6740855895.46,1349926083.00
10,Star Wars: The Last Jedi,6654831607.61,1332698830.00


## Pergunta 5 - Ator atual com mais participações

In [0]:
max_date = (
    df_dim_movies
        .where(F.col("data_lancamento") <= F.current_date())
        .select(F.max("data_lancamento"))
        .collect()[0][0]
)

df_bridge_movie_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_dim_people = spark.table(f"{gold_schema}.dim_people")

df_ator_atual = (
    df_dim_movies
        .where(F.col("status_filme") == "Lançado")
        .where(F.col("data_lancamento") >= F.add_months(F.lit(max_date), -24))
        .join(df_bridge_movie_person, on="sk_movie_id", how="inner")
        .join(df_dim_people, on="sk_person_id", how="inner")
        .where(F.col("tipo_pessoa") == "Ator")
        .groupBy("nome_pessoa")
        .agg(F.count("sk_movie_id").alias("participacoes"))
        .orderBy(F.desc("participacoes"))
        .limit(50)
        .select("nome_pessoa", "participacoes")
)

display(df_ator_atual)

nome_pessoa,participacoes
KEVIN HART,64
JOSH HARTNETT,59
BRANDON H. MORGAN,59
JOHN CENA,59
GREG KRIEK,59
OLIVIA PARK,59
BRANDEN MORGAN,59
BEN SCHWARTZ,59
ALEX DAUPHIN,59
DAVID CHAN,59


## Pergunta 6 - Produtora com maior lucro

In [0]:
max_date = (
    df_dim_movies
        .where(F.col("data_lancamento") <= F.current_date())
        .select(F.max("data_lancamento"))
        .collect()[0][0]
)

df_dim_companies = spark.table(f"{gold_schema}.dim_companies")
df_bridge_movie_company = spark.table(f"{gold_schema}.bridge_movie_company")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_fact_movies_performance = spark.table(f"{gold_schema}.fact_movies_performance")

df_empresas_mais_investidas = (
    df_dim_movies
        .where(F.col("status_filme") == "Lançado")
        .where(F.col("data_lancamento") >= F.add_months(F.lit(max_date), -5 * 12))
        .join(df_bridge_movie_company, on="sk_movie_id", how="inner")
        .join(df_fact_movies_performance, on="sk_movie_id", how="inner")
        .groupBy("sk_company_id")
        .agg(F.sum("lucro_usd").alias("lucro_total"))
        .orderBy(F.desc("lucro_total"))
        .limit(1)
        .join(df_dim_companies, on="sk_company_id", how="inner")
        .select("nome_produtora", "lucro_total")
)

display(df_empresas_mais_investidas)

nome_produtora,lucro_total
UNIVERSAL PICTURES,6116374554.00
